In [1]:
## Standard Stuff
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cf
import matplotlib.pyplot as plt
import dask.array as da
import numcodecs
import cftime

## HEALPix Specific
import healpix as hp
import easygems.healpix as egh
import easygems.remap as egr

import intake     # For catalogs
import zarr       # Data Formatting

import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # don't warn us about future package conflicts

In [4]:
online_cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")["online"]
print(list(online_cat))
print(online_cat["nicam_gl11"])
nicam = online_cat["nicam_gl11"](zoom=6).to_dask()
nicam = nicam.pipe(egh.attach_coords)
nicam

['CERES_EBAF', 'ERA5', 'IR_IMERG', 'JRA3Q', 'MERRA2', 'arp-gem-1p3km', 'arp-gem-2p6km', 'casesm2_10km_nocumulus', 'icon_d3hp003', 'icon_d3hp003aug', 'icon_d3hp003feb', 'icon_ngc4008', 'ifs_tco3999-ng5_deepoff', 'ifs_tco3999-ng5_rcbmf', 'ifs_tco3999-ng5_rcbmf_cf', 'ifs_tco3999_rcbmf', 'nicam_220m_test', 'nicam_gl11', 'scream-dkrz', 'tracking-d3hp003', 'um_Africa_km4p4_RAL3P3_n1280_GAL9_nest', 'um_CTC_km4p4_RAL3P3_n1280_GAL9_nest', 'um_SAmer_km4p4_RAL3P3_n1280_GAL9_nest', 'um_SEA_km4p4_RAL3P3_n1280_GAL9_nest', 'um_glm_n1280_CoMA9_TBv1p2', 'um_glm_n1280_GAL9', 'um_glm_n2560_RAL3p3']
sources:
  nicam_gl11:
    args:
      chunks: null
      consolidated: true
      urlpath: https://nowake.nicam.jp/files/healpix/NICAM_2d3h_z0.zarr
    description: ''
    driver: intake_xarray.xzarr.ZarrSource
    metadata:
      catalog_dir: https://digital-earths-global-hackathon.github.io/catalog/online
      experiment_id: atm_only
      project: global_hackathon
      references: https://doi.org/10.1029

<xarray.Dataset> Size: 19GB
Dimensions:    (time: 2920, cell: 49152, bnds: 2)
Coordinates:
    lev        float64 8B 0.0
  * time       (time) datetime64[ns] 23kB 2020-03-01T01:30:00 ... 2021-02-28T...
    crs        int64 8B 0
  * cell       (cell) int64 393kB 0 1 2 3 4 5 ... 49147 49148 49149 49150 49151
    lat        (cell) float64 393kB 0.5968 1.194 1.194 ... -1.194 -1.194 -0.5968
    lon        (cell) float64 393kB 45.0 45.7 44.3 45.0 ... 315.7 314.3 315.0
Dimensions without coordinates: bnds
Data variables: (12/31)
    clivi      (time, cell) float32 574MB ...
    clt        (time, cell) float64 1GB ...
    clwvi      (time, cell) float32 574MB ...
    hflsd      (time, cell) float32 574MB ...
    hfssd      (time, cell) float32 574MB ...
    huss       (time, cell) float32 574MB ...
    ...         ...
    sftlf      (cell) float32 197kB ...
    tas        (time, cell) float64 1GB ...
    time_bnds  (time, bnds) datetime64[ns] 47kB ...
    ts         (time, cell) float64 1GB ...
    uas        (time, cell) float64 1GB ...
    vas        (time, cell) float64 1GB ...

In [9]:
nicam = xr.open_dataset("https://nowake.nicam.jp/files/healpix/NICAM_2d3h_z7.zarr", consolidated=True, engine="zarr")
nicam.to_zarr("/scratch/cimes/cs3554/NICAM_2d3h_z7.zarr", consolidated=True)

ClientPayloadError: Response payload is not completed: <TransferEncodingError: 400, message='Not enough data for satisfy transfer length header.'>. ConnectionResetError(104, 'Connection reset by peer')

In [2]:
online_cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")["online"]
print(list(online_cat))

['CERES_EBAF', 'ERA5', 'IR_IMERG', 'JRA3Q', 'MERRA2', 'arp-gem-1p3km', 'arp-gem-2p6km', 'casesm2_10km_nocumulus', 'icon_d3hp003', 'icon_d3hp003aug', 'icon_d3hp003feb', 'icon_ngc4008', 'ifs_tco3999-ng5_deepoff', 'ifs_tco3999-ng5_rcbmf', 'ifs_tco3999-ng5_rcbmf_cf', 'ifs_tco3999_rcbmf', 'nicam_220m_test', 'nicam_gl11', 'scream-dkrz', 'tracking-d3hp003', 'um_Africa_km4p4_RAL3P3_n1280_GAL9_nest', 'um_CTC_km4p4_RAL3P3_n1280_GAL9_nest', 'um_SAmer_km4p4_RAL3P3_n1280_GAL9_nest', 'um_SEA_km4p4_RAL3P3_n1280_GAL9_nest', 'um_glm_n1280_CoMA9_TBv1p2', 'um_glm_n1280_GAL9', 'um_glm_n2560_RAL3p3']


In [9]:
for model_name in list(online_cat):
    try:
        model = online_cat[model_name].to_dask()
        model = model.pipe(egh.attach_coords)
        try:
            rsut = model.rsut
            rsutcs = model.rsutcs
            rsdt = model.rsdt
            try:
                rsds = model.rsds
                rsus = model.rsus
                rsdscs = model.rsdscs
                rsuscs = model.rsuscs
            except:
                print(model_name+" doesn't have all surface fluxes")
        except:
            print(model_name+" doesn't have all toa fluxes")
    except: 
        print("can't open "+model_name)

CERES_EBAF doesn't have all toa fluxes
can't open ERA5
IR_IMERG doesn't have all toa fluxes
JRA3Q doesn't have all toa fluxes
can't open MERRA2
can't open arp-gem-1p3km
can't open arp-gem-2p6km
casesm2_10km_nocumulus doesn't have all toa fluxes
icon_d3hp003aug doesn't have all toa fluxes
icon_d3hp003feb doesn't have all toa fluxes
icon_ngc4008 doesn't have all toa fluxes
ifs_tco3999-ng5_deepoff doesn't have all toa fluxes
ifs_tco3999-ng5_rcbmf doesn't have all toa fluxes
ifs_tco3999-ng5_rcbmf_cf doesn't have all toa fluxes
ifs_tco3999_rcbmf doesn't have all toa fluxes
can't open nicam_220m_test
scream-dkrz doesn't have all toa fluxes
can't open tracking-d3hp003
um_Africa_km4p4_RAL3P3_n1280_GAL9_nest doesn't have all surface fluxes
um_CTC_km4p4_RAL3P3_n1280_GAL9_nest doesn't have all surface fluxes
um_SAmer_km4p4_RAL3P3_n1280_GAL9_nest doesn't have all surface fluxes
um_SEA_km4p4_RAL3P3_n1280_GAL9_nest doesn't have all surface fluxes
um_glm_n1280_CoMA9_TBv1p2 doesn't have all surface fl

In [23]:
for model_name in list(online_cat):
    try:
        model = online_cat[model_name].to_dask()
        model = model.pipe(egh.attach_coords)
    except:
        print("can't open "+model_name)

    try:
        rsutcs = model.rsutcs
        rsuscs = model.rsuscs
        rsdt = model.rsdt
        print("SUCCESS!! with "+model_name)
        print(list(model.variables))
    except:
        # print(model_name)
        # print(list(model.variables))
        x = 3

can't open ERA5
can't open MERRA2
can't open arp-gem-1p3km
SUCCESS!! with arp-gem-1p3km
['clivi', 'clt', 'crs', 'healpix', 'hfls', 'hfss', 'huss', 'lwp', 'pr', 'prw', 'ps', 'psl', 'rlds', 'rldscs', 'rlus', 'rluscs', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'tas', 'tauu', 'tauv', 'time', 'ts', 'uas', 'vas']
can't open arp-gem-2p6km
SUCCESS!! with arp-gem-2p6km
['clivi', 'clt', 'crs', 'healpix', 'hfls', 'hfss', 'huss', 'lwp', 'pr', 'prw', 'ps', 'psl', 'rlds', 'rldscs', 'rlus', 'rluscs', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'tas', 'tauu', 'tauv', 'time', 'ts', 'uas', 'vas']
SUCCESS!! with icon_d3hp003
['clivi', 'clt', 'clwvi', 'egpvi', 'einvi', 'ekhvi', 'ekvvi', 'hflsd', 'hfssd', 'hur', 'hus', 'huss', 'mrso', 'o3vi', 'orog', 'pr', 'pressure', 'pressure_rva', 'prs', 'prw', 'ps', 'psl', 'qall', 'rlds', 'rldscs', 'rlus', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'rva', 'sft

In [34]:
um = online_cat["arp-gem-2p6km"].to_dask().pipe(egh.attach_coords)
um

KeyError: "Receive multiple variables for key 'grid_mapping': ['crs', 'healpix']. Expected only one. Please pass a list ['grid_mapping'] instead to get all variables matching 'grid_mapping'."